# Create a mock galaxy to be fit with Bagpipes and Prospector

In [ ]:
import numpy as np 
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True

from astropy.cosmology import WMAP9 as cosmo
import bagpipes as pipes
import matplotlib.pyplot as plt
#%matplotlib inline

from astropy.io import fits
from astropy.table import Table

Set the parameters for the galaxy

In [ ]:

fit_instructions = {}                       # The fit instructions dictionary

# Nebular component
nebular = {}
nebular["logU"] = -3.0                # Log_10 of the ionisation parameter.

exp = {}                          # Tau model star formation history component
exp["age"] = 2.2                 # Gyr
exp["tau"] = 0.75                 # Gyr
exp["massformed"] = 10.5            # log_10(M*/M_solar)
exp["metallicity"] = -0.5          # Z/Z_oldsolar

zred = 2.50 # Set custom redshift

# Dust absorption parameters
dust = {}                               # Dust component
dust["type"] = "CF00"                   # Define the shape of the attenuation curve
dust["Av"] = 1.2                  # Vary Av between 0 and 3.2 magnitudes
dust["n"] = 0.5                 # Vary the slope of the attenuation curve from -1.0 to 1.5

# Dust emission parameters (now free parameters)
dust["qpah"] = 2.0                  # PAH mass fraction (spanning the entire grid)
dust["umin"] = 5.0                   # Lower limit of starlight intensity distribution (spanning the entire grid)
dust["gamma"] = 0.1                   # Fraction of stars at Umin

model_components = {}                   # The model components dictionary
model_components["nebular"] = nebular
model_components["redshift"] = zred      # Observed redshift  
model_components["exponential"] = exp   
model_components["dust"] = dust

bluejay_filt_list = np.loadtxt("filters/bluejay_filt_list_nomiri.txt", dtype="str")
model = pipes.model_galaxy(model_components, filt_list=bluejay_filt_list, phot_units="mujy")

In [ ]:
fig = model.plot()
fig = model.sfh.plot()

In [ ]:
model.plot_full_spectrum()

# Get model photometry

In [ ]:
from json import load


true_flux = model.photometry # This is your 'noiseless' truth

flux_err = true_flux * 0.1  # Add 10% flux error and perturb the observations
mock_flux = np.random.normal(loc=true_flux, scale=flux_err)

# Quick diagnostic printout
for i in range(len(true_flux)):
    print(f"Filter {i:02d} | True: {true_flux[i]:.2e} | Mock Observed: {mock_flux[i]:.2e} +/- {flux_err[i]:.2e}")
    
photometry = np.c_[mock_flux, flux_err]

def load_mock_data(objid):
    print(f"Loading data for mock object {objid}")
    return photometry

# Fit the mock galaxy with Bagpipes

In [ ]:
# Redshift
def get_zred(galaxy_id):    
    # --- Read Blue Jay catalogue ---
    blue = "/Users/benjamincollins/University/Master/BlueJay/BlueJay_sample.txt"
    tbl = Table.read(blue, format="ascii.basic")
    
    row = tbl[tbl['id'] == int(galaxy_id)]
    
    # Make sure that the code doesn't crash if it can't find the ID in the catalogue
    if len(row) == 0:   
        return None, False
    
    z_spec = row['z_spec'][0]
    
    if z_spec is not None and not np.isnan(z_spec):
        return z_spec, True
    else:
        z_phot = row['z_phot'][0]
        return z_phot, False

# Star formation histories
def zred_to_agebins(zred, z_limit_sfh=20.0, nbins_sfh=8):
    tuniv = cosmo.age(zred).value*1e9   # Age of the universe at the observed redshift in years
    #tbinmax = tuniv-cosmo.age(z_limit_sfh).value*1e9 # Maximum age bin edge corresponding to z_limit_sfh
    tbinmax = tuniv*0.95
    # Compute edges in logarithmic space
    log_edges = np.append(np.array([0.0, 6.7, 7.0]), np.linspace(7.0, np.log10(tbinmax), int(nbins_sfh-1))[1:])
    bin_edges = 10**log_edges   # Convert back to linear space
    bin_edges /= 1e6 # ensure that edges are in Myr for Bagpipes
    return bin_edges.tolist()   # return list of age bin edges

fit_instructions = {}                       # The fit instructions dictionary

# Nebular component
nebular = {}
nebular["logU"] = (-4., -1.)                # Log_10 of the ionisation parameter.
#nebular["fesc"] = (0., 1.)                  # IMPORTANT: Escape fraction of ionising photons. Standard value is 0.1

# Dust absorption parameters
dust = {}                               # Dust component
dust["type"] = "CF00"                   # Define the shape of the attenuation curve
dust["Av"] = (0., 4.0)                  # Vary Av between 0 and 3.2 magnitudes
dust["n"] = (-1.0, 1.5)                 # Vary the slope of the attenuation curve from -1.0 to 1.5
dust["n_prior"] = "Gaussian"            # Set a Gaussian prior
dust["n_prior_mu"] = 0.7                # Centred on the standard CF00 slope of 0.7
dust["n_prior_sigma"] = 0.3             # With a width of 0.3

# Dust emission parameters (now free parameters)
dust["qpah"] = (0.1, 4.58)                  # PAH mass fraction (spanning the entire grid)
dust["umin"] = (0.1, 25.)                   # Lower limit of starlight intensity distribution (spanning the entire grid)
dust["gamma"] = (0., 1.0)                   # Fraction of stars at Umin

# Alternatively use a very narrow Gaussian prior centred on the spectroscopic redshift:
zred = 2.50
fit_instructions["redshift"] = (zred-1, zred+1)     # Set the redshift prior to vary within +/1 of the spectroscopic redshift
fit_instructions["redshift_prior"] = "Gaussian"     # Use a Gaussian prior
fit_instructions["redshift_prior_mu"] = zred        # Centred on the spectroscopic redshift
fit_instructions["redshift_prior_sigma"] = 0.005    # With a very narrow width

# Setting the SFH priors
continuity = {}
continuity["massformed"] = (8.5, 13)            # Log10 of solar mass formed
continuity["metallicity"] = (0.01, 3.16)        # Linear Z/Z_solar: 0.01 to 3.16
continuity["metallicity_prior"] = "log_10"      # Logarithmic prior for metallicity

# Define the dsfr ratios
for i in range(1, 7):   # for 7 bins
    continuity[f"dsfr{i}"] = (-10., 10.) 
    continuity[f"dsfr{i}_prior"] = "student_t"
    #continuity["dsfr" + str(i) + "_prior_df"] = 2       # Default nu value of 2 in the student_t distribution
    #continuity["dsfr" + str(i) + "_prior_scale"] = 0.3  # Default sigma value of 0.3 in the student_t distribution

fit_instructions["dust"] = dust
fit_instructions["nebular"] = nebular
fit_instructions["continuity"] = continuity

print(fit_instructions)

In [ ]:
run = "mock_fit"

# Create galaxy object for the current ID
galaxy = pipes.galaxy(f"9999", load_data=load_mock_data, spectrum_exists=False, filt_list=bluejay_filt_list)

# Calculate the agebins for the star formation history of the galaxy based on its redshift and update the fit instructions
age_bins = zred_to_agebins(zred, nbins_sfh=7)
print("Age bins:", age_bins)
fit_instructions["continuity"]["bin_edges"] = age_bins   

fit = pipes.fit(galaxy, fit_instructions, run=run)

fit.fit(verbose=True, n_live=600, sampler='nautilus', pool=6, discard_exploration=True)

fig = fit.plot_spectrum_posterior(save=True, show=False)
fig = fit.plot_sfh_posterior(save=True, show=False)
fig = fit.plot_corner(save=True, show=False) 

In [ ]:
fig = fit.plot_spectrum_posterior(save=False, show=True)
fig = fit.plot_sfh_posterior(save=False, show=True)
fig = fit.plot_corner(save=False, show=True)